# C1 + C2 — Playground interactif

Lance le serveur avant d'utiliser les cellules HTTP (`uvicorn app.main:app --reload --port 8000`)  
ou utilise les classes directement (sections "sans serveur").

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# ── Mets tes clés ici (lit aussi backend/.env automatiquement) ─────────────
from dotenv import load_dotenv
load_dotenv(os.path.join(os.path.dirname('..'), '.env'))

# Ou surcharge manuellement :
# os.environ['ANTHROPIC_API_KEY']         = 'sk-ant-...'
# os.environ['SUPABASE_SERVICE_ROLE_KEY']  = 'eyJ...'
# os.environ['COMPOSIO_API_KEY']           = 'ak_...'
# ──────────────────────────────────────────────────────────────────────────

print('Env OK — ANTHROPIC_API_KEY set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

---
## C1 — Catalogue & Connexions

### 1. Catalogue statique

In [ ]:
from app.control.catalog import get_catalog, get_catalog_item

catalog = get_catalog()
print(f'{len(catalog)} apps disponibles:\n')
for item in catalog:
    print(f'  {item.icon}  {item.display_name:20s} [{item.app}]  — {item.description}')

In [ ]:
# Lookup d'une app spécifique
item = get_catalog_item('strava')
print(item.model_dump_json(indent=2))

### 2. Composio client (mode mock)

In [ ]:
from app.connections.composio_client import (
    initiate_connection,
    get_connection_status,
    get_mcp_url,
    get_available_tools,
)

for app in ['strava', 'gmail', 'github']:
    result = initiate_connection(app)
    status = get_connection_status(result.account_id)
    mcp = get_mcp_url(app, result.account_id)
    tools = get_available_tools(app, result.account_id)
    print(f'{app:15s}  account={result.account_id}  redirect={result.redirect_url}')
    print(f'  status={status.value}  mcp={mcp}')
    print(f'  tools={tools}\n')

### 3. MCP exposer

In [ ]:
from app.contracts import Connection, ConnectionStatus
from app.connections.mcp_exposer import get_mcp_block, filter_tools

conn = Connection(
    id='demo-conn',
    user_id='demo-user',
    app='strava',
    status=ConnectionStatus.CONNECTED,
    mcp_url='http://localhost:9000/mcp/strava',
    available_tools=['get_activities', 'get_activity_detail', 'get_athlete'],
)

block = get_mcp_block(conn)
print('MCP block:', block)

# Filtre un sous-ensemble
filtered = filter_tools(conn, ['get_activities', 'nonexistent_tool'])
print('Filtered tools:', filtered)

### 4. Via HTTP — endpoints C1 (nécessite le serveur)

Lance d'abord : `uvicorn app.main:app --reload --port 8000`

In [ ]:
import httpx, json

BASE = 'http://localhost:8000'

try:
    r = httpx.get(f'{BASE}/catalog', timeout=3)
    apps = r.json()['apps']
    print(f'GET /catalog → {len(apps)} apps')
    for a in apps:
        print(f"  {a['icon']}  {a['display_name']}")
except httpx.ConnectError:
    print('⚠️  Serveur non démarré — lance : uvicorn app.main:app --reload --port 8000')

In [ ]:
try:
    r = httpx.get(f'{BASE}/connections', timeout=3)
    print('GET /connections:', json.dumps(r.json(), indent=2))
except httpx.ConnectError:
    print('⚠️  Serveur non démarré')

In [ ]:
try:
    r = httpx.post(f'{BASE}/connections', json={'app': 'strava'}, timeout=3)
    print('POST /connections:', json.dumps(r.json(), indent=2))
except httpx.ConnectError:
    print('⚠️  Serveur non démarré')

---
## C2 — Hermes Writer

### 5. HermesWriter — écriture d'un profil

In [ ]:
import asyncio
from pathlib import Path
from app.mocks.mock_bundle import get_mock_bundle
from app.mocks.mock_connections import get_mock_connections
from app.provisioning.hermes_writer import HermesWriter

# Écrit dans un dossier temporaire
tmp_dir = Path('/tmp/jarvis_notebook_profiles')
tmp_dir.mkdir(exist_ok=True)

bundle = get_mock_bundle()
connections = get_mock_connections()

writer = HermesWriter(profiles_dir=tmp_dir)
handle = await writer.write(bundle, connections)

print('ProfileHandle:')
print(handle.model_dump_json(indent=2))
print('\nFichiers créés:')
for f in sorted((tmp_dir / 'demo-project').iterdir()):
    print(f'  {f.name}')

In [ ]:
# Inspecte chaque fichier
profile_dir = tmp_dir / 'demo-project'
for filename in ['SOUL.md', 'USER.md', 'MEMORY.md', 'config.yaml']:
    content = (profile_dir / filename).read_text()
    print(f'\n──── {filename} ────')
    print(content)

### 6. _build_mcp_yaml — bloc MCP à partir des connexions

In [ ]:
from app.provisioning.hermes_writer import _build_mcp_yaml
from app.contracts import Connection, ConnectionStatus

connections = [
    Connection(
        id='c1', user_id='u1', app='strava',
        status=ConnectionStatus.CONNECTED,
        mcp_url='http://localhost:9000/mcp/strava',
        available_tools=['get_activities', 'get_activity_detail'],
    ),
    Connection(
        id='c2', user_id='u1', app='gmail',
        status=ConnectionStatus.CONNECTED,
        mcp_url='http://localhost:9000/mcp/gmail',
        available_tools=['list_emails', 'send_email'],
    ),
    Connection(
        id='c3', user_id='u1', app='notion',
        status=ConnectionStatus.PENDING,  # pas encore connecté
        mcp_url=None,
    ),
]

print(_build_mcp_yaml(connections))

---
## C2 — Ingestion

### 7. Jobs de scraping (mode mock)

In [ ]:
from app.ingestion.jobs import strava, gmail, apple_health
from app.contracts import Connection, ConnectionStatus

mock_conn = lambda app: Connection(
    id=f'conn-{app}', user_id='demo-user', app=app,
    status=ConnectionStatus.CONNECTED,
    mcp_url=f'http://localhost:9000/mcp/{app}',
    available_tools=[],
)

for module, app in [(strava, 'strava'), (gmail, 'gmail'), (apple_health, 'apple_health')]:
    raw = await module.scrape(mock_conn(app))
    print(f'\n══ {app} ══')
    print(raw)

### 8. Summarizer — appel Claude (nécessite ANTHROPIC_API_KEY)

In [ ]:
from app.ingestion.summarizer import Summarizer
import anthropic

sample_data = """
Recent Strava activities:
- Morning Run: 5.2 km in 26 min
- Long Run: 10.1 km in 52 min
- Easy Ride: 32 km in 1h10
Weekly mileage: ~22 km
Average pace last 4 runs: 5:05/km
"""

try:
    summarizer = Summarizer()
    facts = await summarizer.summarize(sample_data, 'strava')
    print(f'{len(facts)} faits extraits:')
    for fact in facts:
        print(f'  {fact}')
except anthropic.AuthenticationError:
    print('⚠️  ANTHROPIC_API_KEY invalide — mets ta vraie clé dans la cellule 1')
    print('   (Les autres cellules fonctionnent sans clé)')

### 9. Ingestor complet (DB mock)

In [ ]:
from unittest.mock import MagicMock, AsyncMock
from app.ingestion.ingestor import Ingestor
from app.contracts import Connection, ConnectionStatus

# Mock DB qui loggue les writes
written = []
mock_db = MagicMock()
def fake_insert(project_id, kind, content):
    written.append({'project_id': project_id, 'kind': kind, 'content': content})
    return {'id': f'mem-{len(written)}'}

import app.db.queries as q
original_insert = q.insert_memory
q.insert_memory = fake_insert

# Mock summarizer rapide (sans appel Claude)
fast_summarizer = MagicMock()
fast_summarizer.summarize = AsyncMock(return_value=[
    '- User runs ~22 km per week',
    '- Average pace 5:05/km',
    '- Also cycles occasionally',
])

strava_conn = Connection(
    id='c1', user_id='demo-user', app='strava',
    status=ConnectionStatus.CONNECTED,
    mcp_url='http://localhost:9000/mcp/strava',
    available_tools=['get_activities'],
)

ingestor = Ingestor(db=mock_db, summarizer=fast_summarizer)
await ingestor.ingest('demo-project', [strava_conn])

print(f'{len(written)} entrées mémoire écrites:')
for entry in written:
    print(f"  [{entry['kind']}] {entry['content']}")

# Restore
q.insert_memory = original_insert

---
## C2 — State Machine

### 10. Transitions FSM (DB mock)

In [ ]:
from unittest.mock import MagicMock
from app.provisioning.state_machine import transition, TRANSITIONS
from app.contracts import ProvisioningState

# Visualise toutes les transitions valides
print('Transitions valides:')
for from_state, to_states in TRANSITIONS.items():
    arrow = ' → '.join(s.value for s in to_states) if to_states else '(terminal)'
    print(f'  {from_state.value:15s} → {arrow}')

In [ ]:
# Simule un run complet avec mocks
from app.provisioning.state_machine import run_provisioning
from app.contracts import ProfileHandle
from app.mocks.mock_bundle import get_mock_bundle

states_seen = []
mock_db = MagicMock()
mock_db.table.return_value.update.return_value = mock_db.table.return_value
mock_db.table.return_value.eq.return_value = mock_db.table.return_value
mock_db.table.return_value.execute.return_value = MagicMock(data=[{'id': 'proj-1', 'status': 'draft'}])

# Capture les transitions
original_update = mock_db.table.return_value.update
def track_update(payload):
    states_seen.append(payload.get('status', '?'))
    return mock_db.table.return_value
mock_db.table.return_value.update = track_update

mock_provisioner = MagicMock()
mock_provisioner.provision = AsyncMock(return_value=ProfileHandle(
    project_id='demo-project',
    gateway_url='http://localhost:8080',
    gateway_key='mock-key',
    session_key='agent:demo-project:user:demo-user',
    runtime_key='slot-a',
    status=ProvisioningState.READY,
))

bundle = get_mock_bundle()
handle = await run_provisioning(mock_db, 'demo-project', bundle, [], mock_provisioner)

print('Transitions parcourues:', ' → '.join(states_seen))
print('Handle final:', handle.model_dump_json(indent=2))

---
## Pipeline complet C1 → C2

### 11. Simulation du parcours Sports Coach

Connecte Strava → écrit le profil → ingère la mémoire

In [ ]:
from pathlib import Path
from unittest.mock import MagicMock, AsyncMock
from app.connections.composio_client import initiate_connection, get_mcp_url, get_available_tools
from app.contracts import Connection, ConnectionStatus
from app.mocks.mock_bundle import get_mock_bundle
from app.provisioning.hermes_writer import HermesWriter
from app.ingestion.ingestor import Ingestor

# ── 1. C1: initier la connexion Strava ──
result = initiate_connection('strava')
print(f'[C1] Connexion initiée: account_id={result.account_id}, redirect={result.redirect_url}')

conn = Connection(
    id='demo-strava', user_id='demo-user', app='strava',
    status=ConnectionStatus.CONNECTED,
    mcp_url=get_mcp_url('strava', result.account_id),
    available_tools=get_available_tools('strava', result.account_id),
)
print(f'[C1] Connection: {conn.model_dump_json(indent=2)}')

# ── 2. C2: écrire le profil ──
tmp_dir = Path('/tmp/jarvis_pipeline')
tmp_dir.mkdir(exist_ok=True)
bundle = get_mock_bundle()

writer = HermesWriter(profiles_dir=tmp_dir)
handle = await writer.write(bundle, [conn])
print(f'\n[C2] Profil écrit: {list(p.name for p in (tmp_dir / "demo-project").iterdir())}')

# ── 3. C2: ingestion mémoire (summarizer mocké) ──
mock_db = MagicMock()
memory_log = []

import app.db.queries as q
orig = q.insert_memory
q.insert_memory = lambda pid, kind, content: memory_log.append(content) or {}

fast_summarizer = MagicMock()
fast_summarizer.summarize = AsyncMock(return_value=[
    '- User runs ~22 km per week at 5:05/km pace',
    '- Also does occasional cycling',
])

ingestor = Ingestor(db=mock_db, summarizer=fast_summarizer)
await ingestor.ingest('demo-project', [conn])
q.insert_memory = orig

print(f'\n[C2] Mémoire ingérée ({len(memory_log)} faits):')
for m in memory_log:
    print(f'  {m}')

print('\n✅ Pipeline C1 → C2 complet')